# Primary readout

Conversion rate for the three pre-registered comparisons (Mens vs No E-Mail, Womens vs No E-Mail, Mens vs Womens), Holm corrected, followed by the spend readout. Methods are fixed in `docs/experiment_design.md` and `docs/Log.md` before this notebook was run:
- Wald (unpooled) CI as the main interval, Newcombe hybrid score as a cross-check
- CIs at alpha/3 (98.33%) as the main interval, 95% shown as context
- z-test uses the same unpooled SE as the CI, pooled SE shown once as a cross-check

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from scipy import stats
from load import load_hillstrom

df = load_hillstrom()
TABLES = Path.cwd().parent / "results" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

print(f"rows: {len(df)}")

rows: 64000


## 0. Pre-check, are there enough conversions for the Wald interval?

The Wald interval leans on a normal approximation, which gets shaky when an arm has few events. Before splitting any outcome by arm, checking how many conversions each arm should have using only pooled numbers:

expected conversions in an arm = pooled conversion rate x arm size

The pooled rate is the same 0.9031% validated in `00_eda_pre_registration.ipynb`. Arm sizes are counts of `segment`. Neither splits an outcome by arm.

In [2]:
pooled_rate = df["conversion"].mean()
total_conversions = df["conversion"].sum()
arm_sizes = df["segment"].value_counts().sort_index()
expected_conversions = pooled_rate * arm_sizes
expected_non_conversions = (1 - pooled_rate) * arm_sizes

print(f"pooled conversion rate: {pooled_rate:.4%}")
print(f"total conversions (all arms pooled): {total_conversions:,}")
print()
pd.DataFrame({
    "arm_size": arm_sizes,
    "expected_conversions": expected_conversions,
    "expected_non_conversions": expected_non_conversions,
})

pooled conversion rate: 0.9031%
total conversions (all arms pooled): 578



,arm_size,expected_conversions,expected_non_conversions
segment,,,
Mens E-Mail,21307,192.428844,21114.571156
No E-Mail,21306,192.419812,21113.580187
Womens E-Mail,21387,193.151344,21193.848656


## 1. Conversion by arm

From here on outcomes are split by arm, which the pre-registration commits allow. For each arm, the customer count n, the number who converted, and the conversion rate p = conversions / n.

In [3]:
from statsmodels.stats.proportion import confint_proportions_2indep
from statsmodels.stats.multitest import multipletests

ALPHA = 0.05
N_COMPARISONS = 3
ALPHA_ADJ = ALPHA / N_COMPARISONS
MDE = 0.005

by_arm = df.groupby("segment")["conversion"].agg(n="count", conversions="sum")
by_arm["rate"] = by_arm["conversions"] / by_arm["n"]
by_arm["variance_term"] = by_arm["rate"] * (1 - by_arm["rate"]) / by_arm["n"]
by_arm.style.format({"rate": "{:.4%}", "variance_term": "{:.3e}"})

,n,conversions,rate,variance_term
segment,,,,
Mens E-Mail,21307,267,1.2531%,5.808e-07
No E-Mail,21306,122,0.5726%,2.672e-07
Womens E-Mail,21387,189,0.8837%,4.096e-07


## 2. One comparison worked by hand, Mens vs No E-Mail

difference = p_treatment - p_control

unpooled SE = sqrt( p_t(1-p_t)/n_t + p_c(1-p_c)/n_c ), the same SE the Wald interval uses

z = difference / SE, and the two-sided p-value = 2 x P(Z > |z|)

Wald interval = difference +/- z_crit x SE, with z_crit for 98.33% (main) and 95% (context)

In [4]:
t, c = by_arm.loc["Mens E-Mail"], by_arm.loc["No E-Mail"]

diff = t["rate"] - c["rate"]
se = np.sqrt(t["variance_term"] + c["variance_term"])
z = diff / se
p = 2 * stats.norm.sf(abs(z))
z_crit_adj = stats.norm.ppf(1 - ALPHA_ADJ / 2)
z_crit_95 = stats.norm.ppf(1 - ALPHA / 2)

print(f"p_t {t['rate']:.6f}, p_c {c['rate']:.6f}")
print(f"difference {diff:.6f} ({diff*100:.4f} pp)")
print(f"unpooled SE {se:.6f}")
print(f"z {z:.4f}")
print(f"raw two-sided p {p:.3e}")
print(f"z_crit 98.33% {z_crit_adj:.4f}, z_crit 95% {z_crit_95:.4f}")
print(f"98.33% Wald CI ({(diff - z_crit_adj*se)*100:.4f}, {(diff + z_crit_adj*se)*100:.4f}) pp")
print(f"95% Wald CI ({(diff - z_crit_95*se)*100:.4f}, {(diff + z_crit_95*se)*100:.4f}) pp")

p_t 0.012531, p_c 0.005726
difference 0.006805 (0.6805 pp)
unpooled SE 0.000921
z 7.3899
raw two-sided p 1.469e-13
z_crit 98.33% 2.3940, z_crit 95% 1.9600
98.33% Wald CI (0.4601, 0.9010) pp
95% Wald CI (0.5000, 0.8610) pp


### Cross-check, pooled SE

The pooled SE assumes H0 is true, so both arms share one rate p_pool = (conversions_t + conversions_c) / (n_t + n_c):

pooled SE = sqrt( p_pool(1-p_pool) x (1/n_t + 1/n_c) )

The readout uses the unpooled SE, this just shows how much the choice moves the result.

In [5]:
p_pool = (t["conversions"] + c["conversions"]) / (t["n"] + c["n"])
se_pooled = np.sqrt(p_pool * (1 - p_pool) * (1 / t["n"] + 1 / c["n"]))
z_pooled = diff / se_pooled

print(f"p_pool {p_pool:.6f}")
print(f"pooled SE {se_pooled:.6f} vs unpooled SE {se:.6f}, ratio {se_pooled / se:.4f}")
print(f"z pooled {z_pooled:.4f} vs z unpooled {z:.4f}")
print(f"p pooled {2 * stats.norm.sf(abs(z_pooled)):.3e} vs p unpooled {p:.3e}")

p_pool 0.009129
pooled SE 0.000921 vs unpooled SE 0.000921, ratio 1.0006
z pooled 7.3851 vs z unpooled 7.3899
p pooled 1.523e-13 vs p unpooled 1.469e-13


## 3. All three comparisons

Same formulas for every pre-registered comparison. Mens vs Womens is read as Mens minus Womens. The Newcombe hybrid score interval (98.33%) is computed beside the Wald interval as the cross-check.

In [6]:
comparisons = [
    ("Mens E-Mail", "No E-Mail"),
    ("Womens E-Mail", "No E-Mail"),
    ("Mens E-Mail", "Womens E-Mail"),
]

rows = []
for arm_t, arm_c in comparisons:
    t, c = by_arm.loc[arm_t], by_arm.loc[arm_c]
    diff = t["rate"] - c["rate"]
    se = np.sqrt(t["variance_term"] + c["variance_term"])
    z = diff / se
    newc_lo, newc_hi = confint_proportions_2indep(
        t["conversions"], t["n"], c["conversions"], c["n"], method="newcomb", compare="diff", alpha=ALPHA_ADJ
    )
    rows.append({
        "comparison": f"{arm_t} vs {arm_c}",
        "rate_t": t["rate"],
        "rate_c": c["rate"],
        "diff_pp": diff * 100,
        "relative_lift": diff / c["rate"],
        "se_pp": se * 100,
        "z": z,
        "p_raw": 2 * stats.norm.sf(abs(z)),
        "wald_lo_9833_pp": (diff - z_crit_adj * se) * 100,
        "wald_hi_9833_pp": (diff + z_crit_adj * se) * 100,
        "newcombe_lo_9833_pp": newc_lo * 100,
        "newcombe_hi_9833_pp": newc_hi * 100,
        "wald_lo_95_pp": (diff - z_crit_95 * se) * 100,
        "wald_hi_95_pp": (diff + z_crit_95 * se) * 100,
    })
readout = pd.DataFrame(rows)
readout[["comparison", "rate_t", "rate_c", "diff_pp", "relative_lift", "se_pp", "z", "p_raw"]].style.format(
    {"rate_t": "{:.4%}", "rate_c": "{:.4%}", "diff_pp": "{:.4f}", "relative_lift": "{:.2%}", "se_pp": "{:.4f}", "z": "{:.4f}", "p_raw": "{:.3e}"}
)

,comparison,rate_t,rate_c,diff_pp,relative_lift,se_pp,z,p_raw
0,Mens E-Mail vs No E-Mail,1.2531%,0.5726%,0.6805,118.84%,0.0921,7.3899,1.469e-13
1,Womens E-Mail vs No E-Mail,0.8837%,0.5726%,0.3111,54.33%,0.0823,3.7817,1.558e-04
2,Mens E-Mail vs Womens E-Mail,1.2531%,0.8837%,0.3694,41.80%,0.0995,3.7120,2.056e-04


In [7]:
readout[["comparison", "wald_lo_9833_pp", "wald_hi_9833_pp", "newcombe_lo_9833_pp", "newcombe_hi_9833_pp", "wald_lo_95_pp", "wald_hi_95_pp"]].round(4)

,comparison,wald_lo_9833_pp,wald_hi_9833_pp,newcombe_lo_9833_pp,newcombe_hi_9833_pp,wald_lo_95_pp,wald_hi_95_pp
0,Mens E-Mail vs No E-Mail,0.4601,0.9010,0.4619,0.9058,0.5000,0.8610
1,Womens E-Mail vs No E-Mail,0.1142,0.5080,0.1143,0.5116,0.1499,0.4723
2,Mens E-Mail vs Womens E-Mail,0.1312,0.6076,0.1313,0.6106,0.1744,0.5644


## 4. Holm correction, by hand

Holm sorts the m = 3 raw p-values from smallest to largest. The k-th smallest is multiplied by (m - k + 1), so by 3, then 2, then 1. To keep the adjusted values in the same order as the raw ones, each adjusted value is the running maximum of the ones before it, capped at 1. A comparison is significant if its adjusted p-value is below 0.05.

Cross-checked against statsmodels `multipletests(method="holm")`.

In [8]:
order = readout["p_raw"].sort_values().index
holm = pd.DataFrame({"comparison": readout.loc[order, "comparison"], "p_raw": readout.loc[order, "p_raw"]})
holm["multiplier"] = [N_COMPARISONS - k for k in range(N_COMPARISONS)]
holm["p_times_multiplier"] = holm["p_raw"] * holm["multiplier"]
holm["p_holm"] = holm["p_times_multiplier"].cummax().clip(upper=1)

_, p_holm_sm, _, _ = multipletests(readout["p_raw"], alpha=ALPHA, method="holm")
holm["p_holm_statsmodels"] = pd.Series(p_holm_sm, index=readout.index).loc[order]
holm.style.format({"p_raw": "{:.3e}", "p_times_multiplier": "{:.3e}", "p_holm": "{:.3e}", "p_holm_statsmodels": "{:.3e}"})

,comparison,p_raw,multiplier,p_times_multiplier,p_holm,p_holm_statsmodels
0,Mens E-Mail vs No E-Mail,1.469e-13,3,4.408e-13,4.408e-13,4.408e-13
1,Womens E-Mail vs No E-Mail,1.558e-04,2,3.115e-04,3.115e-04,3.115e-04
2,Mens E-Mail vs Womens E-Mail,2.056e-04,1,2.056e-04,3.115e-04,3.115e-04


## 5. Statistical and practical significance

Applying the pre-registered bars to each comparison.
- Statistical, Holm adjusted p below 0.05
- Practical, the estimated difference at least the MDE of 0.5 percentage points

For the two email vs No E-Mail comparisons these feed the ship tiers. Mens vs Womens only matters as the tiebreaker if both emails clear both bars.

In [9]:
readout["p_holm"] = holm["p_holm"].reindex(readout.index)
readout["stat_sig"] = readout["p_holm"] < ALPHA
readout["meets_mde"] = readout["diff_pp"].abs() >= MDE * 100
readout["ci_9833_excludes_zero"] = (readout["wald_lo_9833_pp"] > 0) | (readout["wald_hi_9833_pp"] < 0)

readout[["comparison", "diff_pp", "p_holm", "stat_sig", "meets_mde", "ci_9833_excludes_zero"]].style.format(
    {"diff_pp": "{:.4f}", "p_holm": "{:.3e}"}
)

,comparison,diff_pp,p_holm,stat_sig,meets_mde,ci_9833_excludes_zero
0,Mens E-Mail vs No E-Mail,0.6805,4.408e-13,True,True,True
1,Womens E-Mail vs No E-Mail,0.3111,3.115e-04,True,False,True
2,Mens E-Mail vs Womens E-Mail,0.3694,3.115e-04,True,False,True


In [10]:
readout.to_csv(TABLES / "conversion_readout.csv", index=False)
holm.to_csv(TABLES / "conversion_holm.csv", index=False)
by_arm.to_csv(TABLES / "conversion_by_arm.csv")
print("saved results/tables/conversion_readout.csv, conversion_holm.csv, conversion_by_arm.csv")

saved results/tables/conversion_readout.csv, conversion_holm.csv, conversion_by_arm.csv


## 6. Spend, pre-check. Does B = 5,000 hold for the BCa interval?

The spend CI is BCa (bias-corrected and accelerated, Efron 1987), with the percentile interval shown beside it. B = 5,000 was chosen from a convergence check on percentile CI width, so it needs checking for BCa too. Same setup as that check, pooled spend across all arms, seed 42, CI width at increasing B. Checked at both 95% and 98.33% since the spend CI level is not locked yet, and the further out the tails, the more resamples it takes to pin them down. Pooled only, no arm split.

**BCa, step by step.** With theta_hat the observed statistic and theta*_1..theta*_B the bootstrap replicates:

- bias correction, z0 = Phi^-1( share of replicates below theta_hat ). If the replicates sit centered on theta_hat, z0 = 0.
- acceleration, a = sum(d_i^3) / ( 6 x (sum(d_i^2))^(3/2) ), where d_i = mean of jackknife values - jackknife value i. Jackknife value i is the statistic recomputed with observation i left out. For a mean that is (total - x_i) / (n - 1), so no refitting is needed.
- adjusted percentiles, for each tail z = z_(alpha/2) or z_(1-alpha/2), use the percentile Phi( z0 + (z0 + z) / (1 - a(z0 + z)) ) of the replicates instead of alpha/2 and 1 - alpha/2.

With z0 = 0 and a = 0 this collapses back to the plain percentile interval.

In [11]:
def bca_interval(replicates, theta_hat, jack_values, alpha):
    z0 = stats.norm.ppf(np.mean(replicates < theta_hat))
    d = jack_values.mean() - jack_values
    a = np.sum(d ** 3) / (6 * np.sum(d ** 2) ** 1.5)
    z_lo, z_hi = stats.norm.ppf(alpha / 2), stats.norm.ppf(1 - alpha / 2)
    p_lo = stats.norm.cdf(z0 + (z0 + z_lo) / (1 - a * (z0 + z_lo)))
    p_hi = stats.norm.cdf(z0 + (z0 + z_hi) / (1 - a * (z0 + z_hi)))
    lo, hi = np.percentile(replicates, [100 * p_lo, 100 * p_hi])
    return lo, hi, z0, a, p_lo, p_hi

rng = np.random.default_rng(seed=42)
spend = df["spend"].to_numpy()
n = len(spend)
theta_hat = spend.mean()
jack_pooled = (spend.sum() - spend) / (n - 1)

max_b = 20000
boot_means = np.empty(max_b)
for b in range(max_b):
    boot_means[b] = spend[rng.integers(0, n, size=n)].mean()

rows = []
for level_name, lvl_alpha in [("95%", ALPHA), ("98.33%", ALPHA_ADJ)]:
    prev = None
    for b in [100, 500, 1000, 2000, 5000, 10000, 20000]:
        lo, hi, z0, a, p_lo, p_hi = bca_interval(boot_means[:b], theta_hat, jack_pooled, lvl_alpha)
        width = hi - lo
        rows.append({"level": level_name, "B": b, "z0": z0, "a": a, "p_lo": p_lo, "p_hi": p_hi,
                     "bca_lo": lo, "bca_hi": hi, "bca_width": width,
                     "pct_change_in_width_vs_prev": None if prev is None else (width / prev - 1) * 100})
        prev = width

bca_convergence = pd.DataFrame(rows)
print(f"pooled mean spend (theta_hat): {theta_hat:.6f}")
bca_convergence.round(6)

pooled mean spend (theta_hat): 1.050908


,level,B,z0,a,p_lo,p_hi,bca_lo,bca_hi,bca_width,pct_change_in_width_vs_prev
0,95%,100,-0.100434,0.013565,0.017654,0.964636,0.932251,1.140917,0.208667,NaN
1,95%,500,0.015040,0.013565,0.030062,0.979544,0.929854,1.154937,0.225083,7.867131
2,95%,1000,0.030084,0.013565,0.032116,0.981023,0.936301,1.166429,0.230128,2.241463
3,95%,2000,0.027576,0.013565,0.031766,0.980783,0.935789,1.168744,0.232954,1.228044
4,95%,5000,0.016043,0.013565,0.030195,0.979645,0.938757,1.172531,0.233774,0.352089
5,95%,10000,-0.003259,0.013565,0.027711,0.977614,0.939041,1.171255,0.232214,-0.667517
6,95%,20000,0.000501,0.013565,0.028181,0.978023,0.939011,1.171261,0.232250,0.015464
7,98.33%,100,-0.100434,0.013565,0.005982,0.988298,0.885886,1.154967,0.269081,NaN
8,98.33%,500,0.015040,0.013565,0.011024,0.993885,0.915033,1.174822,0.259789,-3.453062
9,98.33%,1000,0.030084,0.013565,0.011899,0.994403,0.915710,1.191879,0.276169,6.305158


In [12]:
bca_convergence.to_csv(TABLES / "spend_bca_convergence.csv", index=False)
print("saved results/tables/spend_bca_convergence.csv")

saved results/tables/spend_bca_convergence.csv


## 7. Spend, pre-check. What spend lift can the data detect?

Before spend is split by arm, the smallest difference in mean spend per customer the data could reliably detect, at 90% power (the locked power target), for the two candidate CI levels. Same relationship as the conversion power calc, with the pooled spend standard deviation sigma in place of the proportion variance terms:

detectable difference = (z_(alpha/2) + z_power) x sigma x sqrt(1/n_a + 1/n_b)

Approximation, stated up front. This assumes both arms share the pooled sigma and that the difference in means is close to normal. Spend is heavily skewed, so it is rougher than the conversion version, but with about 21,000 customers per arm the difference in means should still be close to normal. Pooled spend only, no arm split.

In [13]:
sigma_spend = df["spend"].std(ddof=1)
mean_spend = df["spend"].mean()
n_a, n_b = arm_sizes.nsmallest(2)
se_spend_diff = sigma_spend * np.sqrt(1 / n_a + 1 / n_b)
z_power = stats.norm.ppf(0.90)

print(f"pooled spend mean {mean_spend:.6f}, pooled spend SD {sigma_spend:.6f}")
print(f"two smallest arms {n_a:,} and {n_b:,}")
print(f"SE of a difference in mean spend {se_spend_diff:.6f}")
print(f"z_power (90%) {z_power:.4f}")

spend_power = []
for level_name, lvl_alpha in [("95%", ALPHA), ("98.33%", ALPHA_ADJ)]:
    z_a = stats.norm.ppf(1 - lvl_alpha / 2)
    detectable = (z_a + z_power) * se_spend_diff
    spend_power.append({"level": level_name, "z_alpha_half": z_a, "detectable_diff_dollars": detectable,
                        "detectable_relative_to_pooled_mean": detectable / mean_spend})
spend_power = pd.DataFrame(spend_power)
spend_power.to_csv(TABLES / "spend_detectable_diff.csv", index=False)
spend_power.style.format({"z_alpha_half": "{:.4f}", "detectable_diff_dollars": "${:.4f}", "detectable_relative_to_pooled_mean": "{:.2%}"})

pooled spend mean 1.050908, pooled spend SD 15.036448
two smallest arms 21,306 and 21,307
SE of a difference in mean spend 0.145681
z_power (90%) 1.2816


,level,z_alpha_half,detectable_diff_dollars,detectable_relative_to_pooled_mean
0,95%,1.9600,$0.4722,44.94%
1,98.33%,2.3940,$0.5355,50.95%
